# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [ ]:
# Set the model directory.
model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/diffuser-patrolling/wandb/run-20250515_191045-wlhoasjo/files"

# Set the output file.
# output_file = "output.zarr"

# Set the output file automatically.
output_file = "./" + model_dir.split("/")[-2] + ".zarr"

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [ ]:
# Load arguments from the config file.
config_file = os.path.join(model_dir, "config.yaml")
args = yaml.load(open(config_file), Loader=yaml.FullLoader)

def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            v = value["value"]
            if v == None:
                continue
            if type(v) == bool:
                if v:
                    arglist.append(f"--{key}")
                else:
                    arglist.append(f"--no-{key}")
            else:
                arglist.append(f"--{key}")
                arglist.append(str(v))
    return arglist

def wrap_arg_val(val):
    return {"value": val}

# Set required eval-specific arguments. Do not change these!
args["use_wandb"] = wrap_arg_val(False)
args["use_render"] = wrap_arg_val(False)
args["use_eval"] = wrap_arg_val(True)
args["model_dir"] = wrap_arg_val(model_dir)
args["cuda"] = wrap_arg_val(False)
args["eval_output_file"] = wrap_arg_val(output_file)

In [ ]:
args["eval_episodes"] = wrap_arg_val(2)
args["episode_length"] = wrap_arg_val(200)
args["seed"] = wrap_arg_val(1)

# Evaluation

In [ ]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(args["eval_output_file"]["value"]):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        # p = Process(target=main, args=[convert_args(args)])
        # processes.append(p)
        # p.start()
        main(convert_args(args))


    # VARIANT 1: Diffusion prediction.
    argscopy = deepcopy(args)
    argscopy["experiment_name"] = wrap_arg_val("diffusion-prediction")
    start_process(argscopy)


    # VARIANT 2: No prediction.
    argscopy = deepcopy(args)
    argscopy["experiment_name"] = wrap_arg_val("no-prediction")
    argscopy["diffusion_disable"] = wrap_arg_val(True)
    start_process(argscopy)


    # JOIN ALL PROCESSES
    for p in processes:
        p.join()
    print("All processes finished!")

# Analysis

In [ ]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for plotting
categories = {
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        # Average over all episodes and all timesteps.
        'data': [np.mean(output[experiment]['avg_idleness']) for experiment in output]
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        # Average over all episodes and all timesteps.
        'data': [np.mean(output[experiment]['avg_idleness'][:, -1]) for experiment in output]
    },
}
for category in categories:
    values = categories[category]['data']

    # Create bar chart
    plt.bar(list(output.keys()), values)
    
    title = categories[category]['title']
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel'])
    plt.ylabel(f"Timesteps out of {args['episode_length']['value']}" if "ylabel" not in categories[category] else categories[category]['ylabel'])
    plt.title(f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}")
    plt.show()